In [63]:
import sys
import os

sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.base import BaseEstimator, TransformerMixin

import numpy as np
import nltk

# Data Preprocessing

In [64]:
df1 = pd.read_csv('D:\Spam_Classifier\data\spam_ham_dataset.csv')[['text','label']]
# df2 = pd.read_csv('D:\Spam_Classifier\data\spam.csv')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Rahul\AppData\Local\Temp\ipykernel_15452\3047070491.py:1: SyntaxWarning: invalid escape sequence '\S'
  df1 = pd.read_csv('D:\Spam_Classifier\data\spam_ham_dataset.csv')[['text','label']]


In [65]:
df2 = pd.read_csv('D:\Spam_Classifier\data\spam.csv' ,encoding='latin-1')[['v1','v2']]

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Rahul\AppData\Local\Temp\ipykernel_15452\555317484.py:1: SyntaxWarning: invalid escape sequence '\S'
  df2 = pd.read_csv('D:\Spam_Classifier\data\spam.csv' ,encoding='latin-1')[['v1','v2']]


In [66]:
df1.head()

,text,label
0,Subject: enron methanol ; meter # : 988291\r\n...,ham
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",ham
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",ham
3,"Subject: photoshop , windows , office . cheap ...",spam
4,Subject: re : indian springs\r\nthis deal is t...,ham


In [67]:
df2.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [68]:
print(df1.shape)
print(df2.shape)

(5171, 2)
(5572, 2)


In [69]:
print(df1.isna().sum())
print(df2.isna().sum())


text     0
label    0
dtype: int64
v1    0
v2    0
dtype: int64


In [70]:
print(df1.duplicated().sum())
print(df2.duplicated().sum())



178
403


In [71]:
df1.dtypes

text     object
label    object
dtype: object

In [72]:
df2.dtypes

v1    object
v2    object
dtype: object

In [73]:
df2.rename(columns = {'v1':'label','v2':'text'},inplace = True)

## ### 📊 Data Preprocessing Summary

#### The project utilized two publicly available SMS spam datasets to enhance data diversity and model robustness. The initial datasets contained 5,171 and 5,572 records respectively, each with two primary features: message content and label.

#### To ensure consistency and seamless integration, the following preprocessing steps were performed:

####  Column Standardization:**
####  Column names across both datasets were aligned to a uniform schema: `label` (target variable) and `text` (input feature).

####  Data Integrity Checks:**
  
#### Both datasets were examined for missing values, and no null entries were detected, ensuring completeness of the data.
 
#### Duplicate Removal:**
####  A total of 178 duplicate records were identified in the first dataset and 403 in the second dataset. These duplicates were removed to prevent data leakage and model bias.

#### Data Type Validation:
####  Data types were verified and confirmed to be appropriate for downstream processing, with text features stored as strings and labels correctly formatted.

#### Dataset Merging:

####  After cleaning, the datasets were combined into a single unified dataset to improve training generalization and capture a wider variety of spam patterns.




## Data Cleaning

In [74]:
df1.head()

,text,label
0,Subject: enron methanol ; meter # : 988291\r\n...,ham
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",ham
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",ham
3,"Subject: photoshop , windows , office . cheap ...",spam
4,Subject: re : indian springs\r\nthis deal is t...,ham


In [75]:
print(df1.iloc[0]['text'])

Subject: enron methanol ; meter # : 988291
this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary
flow data provided by daren } .
please override pop ' s daily volume { presently zero } to reflect daily
activity you can obtain from gas control .
this change is needed asap for economics purposes .


In [76]:
import re
def remove_headers(text):
    # remove 'subject': 
    text = re.sub(r"subject.?\n","",text,flags=re.IGNORECASE)
    return text

In [77]:
def remove_header_words(text):
    return re.sub(r'\b(subject|from|to|cc)\b', '', text)

In [78]:
def remove_domain_noise(text):
    noise_words = ['enron', 'meter']
    words = text.split()
    words = [w for w in words if w not in noise_words]
    return " ".join(words)

In [79]:
def remove_noise(text):
    text = re.sub(r"\d+","",text)
    text = re.sub(r"[^\w\s]" , " ",text)
    return text

In [80]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  
    return text.strip()

In [81]:
def clean_email_text(text):
    text = remove_headers(text)
    text = remove_noise(text)
    text = normalize_text(text)
    text = remove_header_words(text)
    text = remove_domain_noise(text)
    return text

In [82]:
df1['cleaned_text'] = df1['text'].apply(clean_email_text)

In [83]:
df1.head()

,text,label,cleaned_text
0,Subject: enron methanol ; meter # : 988291\r\n...,ham,methanol this is a follow up the note i gave y...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",ham,hpl nom for january see attached file hplnol x...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",ham,neon retreat ho ho ho we re around that most w...
3,"Subject: photoshop , windows , office . cheap ...",spam,photoshop windows office cheap main trending a...
4,Subject: re : indian springs\r\nthis deal is t...,ham,re indian springs this deal is book the teco p...


In [84]:
df2.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [85]:
print(df2.iloc[50]['text'])


What you thinked about me. First time you saw me in class.


In [86]:
import re
import string

def clean_sms(text):
    text = text.lower()
    
    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [87]:
df2['cleaned_text'] = df2['text'].apply(clean_sms)

In [88]:
df = pd.concat([df1, df2], ignore_index=True)

In [89]:
df.head()

,text,label,cleaned_text
0,Subject: enron methanol ; meter # : 988291\r\n...,ham,methanol this is a follow up the note i gave y...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",ham,hpl nom for january see attached file hplnol x...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",ham,neon retreat ho ho ho we re around that most w...
3,"Subject: photoshop , windows , office . cheap ...",spam,photoshop windows office cheap main trending a...
4,Subject: re : indian springs\r\nthis deal is t...,ham,re indian springs this deal is book the teco p...


In [90]:
df = df[['label','cleaned_text']]

In [91]:
df.head()

,label,cleaned_text
0,ham,methanol this is a follow up the note i gave y...
1,ham,hpl nom for january see attached file hplnol x...
2,ham,neon retreat ho ho ho we re around that most w...
3,spam,photoshop windows office cheap main trending a...
4,ham,re indian springs this deal is book the teco p...


In [92]:
df.shape

(10743, 2)

In [93]:
df.duplicated().sum()

np.int64(994)

In [94]:
df.drop_duplicates(inplace = True)

In [95]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [96]:
df['label'].unique()

array([0, 1])

In [97]:
df.isna().sum()

label           0
cleaned_text    0
dtype: int64

## Feature Engneering & Pipeline

In [98]:
from src.features import TextStats

In [99]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2)), 'cleaned_text'),
        ('stats', Pipeline([
            ('stats', TextStats()),
            ('scaler', StandardScaler())
        ]), 'cleaned_text')
    ]
)

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LinearSVC(class_weight='balanced'))
])

## Modeling & Evaluating Performance

In [100]:
from sklearn.model_selection import train_test_split

X = df[['cleaned_text']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [101]:
pipeline.fit(X_train , y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('tfidf',
                                                  TfidfVectorizer(max_features=5000,
                                                                  ngram_range=(1,
                                                                               2)),
                                                  'cleaned_text'),
                                                 ('stats',
                                                  Pipeline(steps=[('stats',
                                                                   TextStats()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  'cleaned_text')])),
                ('model', LinearSVC(class_weight='balanced'))])

In [102]:
y_pred = pipeline.predict(X_test)

In [103]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9594871794871795
Precision: 0.897196261682243
Recall: 0.9164677804295943
F1 Score: 0.9067296340023613


## Dumping Model

In [104]:
import pickle

pickle.dump(pipeline, open('D:\Spam_Classifier\model/pipeline.pkl', 'wb'))

<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Rahul\AppData\Local\Temp\ipykernel_15452\4072752828.py:3: SyntaxWarning: invalid escape sequence '\S'
  pickle.dump(pipeline, open('D:\Spam_Classifier\model/pipeline.pkl', 'wb'))


## model building

## Calculating Performance